## Imports and Setup

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from loguru import logger

from src import (
    TREATMENTS, N_TREATMENTS, IDX_TO_TREATMENT, TREATMENT_TO_IDX,
    CONTEXT_FEATURES, reward_oracle,
)
from src.utils import setup_plotting, seed_everything, TREATMENT_COLORS

seed_everything(42)
setup_plotting()
print("Setup complete")

## Load Dataset

In [ ]:
df = pd.read_csv("../data/bandit_dataset.csv")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()


## Basic Dataset Info

In [ ]:
print(f"Total patients: {len(df):,}")
print(f"Features: {len(CONTEXT_FEATURES)}")
print(f"Treatments: {N_TREATMENTS}")
print(f"\nColumn dtypes:")
print(df.dtypes.value_counts())
print(f"\nMissing values: {df.isnull().sum().sum()}")
df.describe().round(3)

## Treatment (Action) Distribution from Logging Policy

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
action_counts = df['action_name'].value_counts()
colors = [TREATMENT_COLORS[t] for t in action_counts.index]
ax1.bar(action_counts.index, action_counts.values, color=colors, edgecolor='white')
ax1.set_title("Action Distribution (Clinical Logging Policy)")
ax1.set_ylabel("Count")
for i, (name, count) in enumerate(action_counts.items()):
    ax1.text(i, count + 100, f"{count/len(df)*100:.1f}%", ha='center', fontsize=10)

# Propensity distribution per treatment
for t in TREATMENTS:
    mask = df['action_name'] == t
    ax2.hist(df.loc[mask, 'propensity'], bins=30, alpha=0.5,
             label=t, color=TREATMENT_COLORS[t])
ax2.set_title("Propensity Score Distribution by Treatment")
ax2.set_xlabel("Propensity")
ax2.set_ylabel("Count")
ax2.legend()

plt.tight_layout()
plt.show()

print("\nAction distribution:")
print(df['action_name'].value_counts().to_string())
print(f"\nPropensity stats:")
print(df.groupby('action_name')['propensity'].describe().round(4))

## Context Feature Distributions

In [ ]:
continuous_feats = ['age', 'bmi', 'hba1c_baseline', 'egfr', 'diabetes_duration',
                    'fasting_glucose', 'c_peptide', 'bp_systolic', 'ldl', 'hdl',
                    'triglycerides', 'alt']

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, feat in enumerate(continuous_feats):
    ax = axes[i]
    ax.hist(df[feat], bins=50, color='#2196F3', edgecolor='white', alpha=0.8)
    ax.set_title(feat)
    ax.axvline(df[feat].mean(), color='red', linestyle='--', linewidth=1, label=f'mean={df[feat].mean():.1f}')
    ax.legend(fontsize=8)

plt.suptitle("Continuous Feature Distributions", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Binary Feature Distributions

In [ ]:
binary_feats = ['cvd', 'ckd', 'nafld', 'hypertension']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, feat in enumerate(binary_feats):
    ax = axes[i]
    counts = df[feat].value_counts().sort_index()
    ax.bar(['No', 'Yes'], counts.values, color=['#90CAF9', '#1565C0'], edgecolor='white')
    ax.set_title(feat.upper())
    pct = df[feat].mean() * 100
    ax.text(1, counts.values[1] + 100, f"{pct:.1f}%", ha='center', fontsize=11, fontweight='bold')

plt.suptitle("Comorbidity Prevalence", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Feature Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(14, 11))
feat_cols = continuous_feats + binary_feats
corr = df[feat_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

# Flag strong correlations
strong = []
for i in range(len(feat_cols)):
    for j in range(i+1, len(feat_cols)):
        r = abs(corr.iloc[i, j])
        if r > 0.5:
            strong.append((feat_cols[i], feat_cols[j], round(corr.iloc[i, j], 3)))
print("Strong correlations (|r| > 0.5):")
for a, b, r in sorted(strong, key=lambda x: -abs(x[2])):
    print(f"  {a:<20} ↔ {b:<20} r={r}")

## Reward Distribution (Observed)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Overall reward distribution
ax1.hist(df['reward'], bins=60, color='#2196F3', edgecolor='white', alpha=0.8)
ax1.axvline(df['reward'].mean(), color='red', linestyle='--', label=f"mean={df['reward'].mean():.3f}")
ax1.set_title("Observed Reward Distribution (All Treatments)")
ax1.set_xlabel("HbA1c Reduction")
ax1.set_ylabel("Count")
ax1.legend()

# Per-treatment reward
for t in TREATMENTS:
    mask = df['action_name'] == t
    ax2.hist(df.loc[mask, 'reward'], bins=40, alpha=0.5,
             label=f"{t} (μ={df.loc[mask, 'reward'].mean():.2f})",
             color=TREATMENT_COLORS[t])
ax2.set_title("Observed Reward by Treatment")
ax2.set_xlabel("HbA1c Reduction")
ax2.legend()

plt.tight_layout()
plt.show()

print("Observed reward stats by treatment:")
print(df.groupby('action_name')['reward'].describe().round(3))

## Counterfactual Reward Analysis (Ground Truth)

In [ ]:
reward_cols = [f'reward_{i}' for i in range(N_TREATMENTS)]
cf = df[reward_cols]
cf.columns = TREATMENTS

print("Counterfactual (expected) reward by treatment:")
print(cf.describe().round(3))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Mean counterfactual rewards
means = cf.mean().sort_values(ascending=True)
colors = [TREATMENT_COLORS[t] for t in means.index]
ax1.barh(means.index, means.values, color=colors, edgecolor='white')
ax1.set_xlabel("Mean Expected Reward (no noise)")
ax1.set_title("Average Counterfactual Reward by Treatment")
for i, (t, v) in enumerate(means.items()):
    ax1.text(v + 0.05, i, f"{v:.3f}", va='center', fontsize=10)

# Distribution of counterfactual rewards
for t in TREATMENTS:
    ax2.hist(cf[t], bins=50, alpha=0.5, label=t, color=TREATMENT_COLORS[t])
ax2.set_title("Counterfactual Reward Distributions")
ax2.set_xlabel("Expected Reward")
ax2.legend()

plt.tight_layout()
plt.show()

## Optimal Action Distribution (Oracle)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Optimal action counts
opt_counts = df['optimal_action_name'].value_counts()
colors = [TREATMENT_COLORS[t] for t in opt_counts.index]
ax1.bar(opt_counts.index, opt_counts.values, color=colors, edgecolor='white')
ax1.set_title("Optimal Treatment Distribution (Oracle)")
ax1.set_ylabel("Count")
for i, (name, count) in enumerate(opt_counts.items()):
    ax1.text(i, count + 100, f"{count/len(df)*100:.1f}%", ha='center', fontsize=10)

# Logging policy vs oracle comparison
log_dist = df['action_name'].value_counts(normalize=True).reindex(TREATMENTS).fillna(0)
opt_dist = df['optimal_action_name'].value_counts(normalize=True).reindex(TREATMENTS).fillna(0)

x = np.arange(N_TREATMENTS)
w = 0.35
ax2.bar(x - w/2, log_dist.values, w, label='Logging Policy', color='#90CAF9', edgecolor='white')
ax2.bar(x + w/2, opt_dist.values, w, label='Oracle Optimal', color='#1565C0', edgecolor='white')
ax2.set_xticks(x)
ax2.set_xticklabels(TREATMENTS)
ax2.set_ylabel("Proportion")
ax2.set_title("Logging Policy vs Oracle — Action Distribution")
ax2.legend()

plt.tight_layout()
plt.show()

# Accuracy of logging policy
accuracy = (df['action'] == df['optimal_action']).mean()
print(f"Logging policy accuracy (matches oracle): {accuracy:.4f} ({accuracy*100:.1f}%)")


## Regret Analysis

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Regret distribution
ax1.hist(df['regret'], bins=60, color='#FF9800', edgecolor='white', alpha=0.8)
ax1.axvline(df['regret'].mean(), color='red', linestyle='--',
            label=f"mean={df['regret'].mean():.3f}")
ax1.axvline(0, color='green', linestyle='-', linewidth=2, label='zero regret')
ax1.set_title("Regret Distribution (Logging Policy)")
ax1.set_xlabel("Regret (Oracle Reward − Observed Reward)")
ax1.legend()

# Regret by logged treatment
regret_by_treatment = df.groupby('action_name')['regret'].mean().sort_values()
colors = [TREATMENT_COLORS[t] for t in regret_by_treatment.index]
ax2.barh(regret_by_treatment.index, regret_by_treatment.values, color=colors, edgecolor='white')
ax2.set_xlabel("Average Regret")
ax2.set_title("Average Regret by Logged Treatment")
for i, (t, v) in enumerate(regret_by_treatment.items()):
    ax2.text(v + 0.01, i, f"{v:.3f}", va='center', fontsize=10)

plt.tight_layout()
plt.show()

print(f"Overall mean regret: {df['regret'].mean():.4f}")
print(f"Zero-regret (correct) decisions: {(df['regret'] == 0).sum()} ({(df['regret'] == 0).mean()*100:.1f}%)")


## Treatment-Context Relationships (Key Clinical Signals)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. BMI vs optimal treatment
ax = axes[0, 0]
for t in TREATMENTS:
    mask = df['optimal_action_name'] == t
    ax.hist(df.loc[mask, 'bmi'], bins=30, alpha=0.5, label=t, color=TREATMENT_COLORS[t])
ax.set_title("BMI by Optimal Treatment")
ax.set_xlabel("BMI")
ax.legend(fontsize=8)

# 2. eGFR vs optimal treatment
ax = axes[0, 1]
for t in TREATMENTS:
    mask = df['optimal_action_name'] == t
    ax.hist(df.loc[mask, 'egfr'], bins=30, alpha=0.5, label=t, color=TREATMENT_COLORS[t])
ax.set_title("eGFR by Optimal Treatment")
ax.set_xlabel("eGFR")
ax.legend(fontsize=8)

# 3. HbA1c baseline vs optimal treatment
ax = axes[0, 2]
for t in TREATMENTS:
    mask = df['optimal_action_name'] == t
    ax.hist(df.loc[mask, 'hba1c_baseline'], bins=30, alpha=0.5, label=t, color=TREATMENT_COLORS[t])
ax.set_title("HbA1c Baseline by Optimal Treatment")
ax.set_xlabel("HbA1c")
ax.legend(fontsize=8)

# 4. Age vs optimal treatment
ax = axes[1, 0]
for t in TREATMENTS:
    mask = df['optimal_action_name'] == t
    ax.hist(df.loc[mask, 'age'], bins=30, alpha=0.5, label=t, color=TREATMENT_COLORS[t])
ax.set_title("Age by Optimal Treatment")
ax.set_xlabel("Age")
ax.legend(fontsize=8)

# 5. C-peptide vs optimal treatment
ax = axes[1, 1]
for t in TREATMENTS:
    mask = df['optimal_action_name'] == t
    ax.hist(df.loc[mask, 'c_peptide'], bins=30, alpha=0.5, label=t, color=TREATMENT_COLORS[t])
ax.set_title("C-peptide by Optimal Treatment")
ax.set_xlabel("C-peptide")
ax.legend(fontsize=8)

# 6. CVD presence vs optimal treatment
ax = axes[1, 2]
cross = pd.crosstab(df['optimal_action_name'], df['cvd'], normalize='columns')
cross.plot(kind='bar', ax=ax, color=['#90CAF9', '#1565C0'], edgecolor='white')
ax.set_title("Optimal Treatment by CVD Status")
ax.set_ylabel("Proportion")
ax.set_xlabel("")
ax.legend(['No CVD', 'CVD'], fontsize=9)
ax.tick_params(axis='x', rotation=45)

plt.suptitle("Clinical Context → Optimal Treatment Mapping", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Counterfactual Reward Heatmap (Treatment × Context Subgroups)

In [ ]:
# Create subgroups
df['bmi_group'] = pd.cut(df['bmi'], bins=[0, 30, 35, 100], labels=['<30', '30-35', '>35'])
df['hba1c_group'] = pd.cut(df['hba1c_baseline'], bins=[0, 8, 10, 100], labels=['<8', '8-10', '>10'])
df['egfr_group'] = pd.cut(df['egfr'], bins=[0, 45, 60, 200], labels=['<45', '45-60', '>60'])
df['age_group'] = pd.cut(df['age'], bins=[0, 50, 65, 100], labels=['<50', '50-65', '>65'])

# Mean counterfactual reward by BMI group
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (group_col, group_name) in zip(axes.flatten(), [
    ('bmi_group', 'BMI Group'),
    ('hba1c_group', 'HbA1c Group'),
    ('egfr_group', 'eGFR Group'),
    ('age_group', 'Age Group'),
]):
    heatmap_data = []
    for g in df[group_col].cat.categories:
        mask = df[group_col] == g
        row = {}
        for k, t in enumerate(TREATMENTS):
            row[t] = df.loc[mask, f'reward_{k}'].mean()
        row['group'] = g
        heatmap_data.append(row)

    hm = pd.DataFrame(heatmap_data).set_index('group')[TREATMENTS]
    sns.heatmap(hm, annot=True, fmt='.2f', cmap='YlGn', ax=ax, linewidths=0.5)
    ax.set_title(f"Expected Reward by {group_name}")
    ax.set_ylabel(group_name)

plt.suptitle("Treatment Effectiveness Across Patient Subgroups", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Reward Separation Analysis (Can the Bandit Learn?)

In [ ]:
# For each patient, compute gap between best and second-best treatment
cf_values = df[[f'reward_{i}' for i in range(N_TREATMENTS)]].values
sorted_cf = np.sort(cf_values, axis=1)[:, ::-1]

best_reward = sorted_cf[:, 0]
second_best = sorted_cf[:, 1]
gap = best_reward - second_best

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(gap, bins=60, color='#4CAF50', edgecolor='white', alpha=0.8)
ax1.axvline(gap.mean(), color='red', linestyle='--', label=f'mean gap={gap.mean():.3f}')
ax1.set_title("Reward Gap: Best − Second Best Treatment")
ax1.set_xlabel("Reward Gap")
ax1.legend()

# What fraction of patients have a clear winner?
thresholds = [0.0, 0.25, 0.5, 1.0, 1.5, 2.0]
fractions = [(gap > t).mean() * 100 for t in thresholds]
ax2.bar([str(t) for t in thresholds], fractions, color='#4CAF50', edgecolor='white')
ax2.set_xlabel("Minimum Gap Threshold")
ax2.set_ylabel("% of Patients")
ax2.set_title("Patients with Clear Treatment Winner")
for i, f in enumerate(fractions):
    ax2.text(i, f + 1, f"{f:.1f}%", ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print(f"Mean reward gap (best vs 2nd best): {gap.mean():.3f}")
print(f"Patients where all treatments within 0.5: {(gap < 0.5).mean()*100:.1f}%")
print(f"Patients with clear winner (gap > 1.0): {(gap > 1.0).mean()*100:.1f}%")

## Propensity Score Validation

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Propensity distribution
ax1.hist(df['propensity'], bins=50, color='#9C27B0', edgecolor='white', alpha=0.8)
ax1.set_title("Overall Propensity Score Distribution")
ax1.set_xlabel("P(action | context)")
ax1.axvline(1/N_TREATMENTS, color='red', linestyle='--', label=f'uniform={1/N_TREATMENTS:.2f}')
ax1.legend()

# Check propensity overlap
ax2.set_title("Propensity Overlap Check")
for t in TREATMENTS:
    mask = df['action_name'] == t
    ax2.hist(df.loc[mask, 'propensity'], bins=30, alpha=0.4,
             label=t, color=TREATMENT_COLORS[t])
ax2.set_xlabel("Propensity Score")
ax2.legend()

plt.tight_layout()
plt.show()

# Verify propensities sum correctly (spot check)
import json
sample_props = df['propensity_all'].head(5).apply(json.loads)
print("Sample propensity vectors (should sum to ~1.0):")
for i, p in enumerate(sample_props):
    print(f"  Patient {i}: {[f'{x:.3f}' for x in p]} → sum={sum(p):.4f}")

print(f"\nPropensity range: [{df['propensity'].min():.4f}, {df['propensity'].max():.4f}]")
print(f"Min propensity > 0: {(df['propensity'] > 0).all()}")


## Summary — Dataset Quality Checklist

In [ ]:
print("=" * 60)
print("DATASET QUALITY CHECKLIST")
print("=" * 60)

checks = {
    "No missing values": df.isnull().sum().sum() == 0,
    "All propensities > 0 (positivity)": (df['propensity'] > 0).all(),
    f"All {N_TREATMENTS} treatments present": df['action'].nunique() == N_TREATMENTS,
    "Counterfactuals available": all(f'reward_{i}' in df.columns for i in range(N_TREATMENTS)),
    "Optimal actions computed": 'optimal_action' in df.columns,
    "Regret computed": 'regret' in df.columns,
    "Meaningful reward separation": gap.mean() > 0.5,
    "Logging policy imperfect (room to improve)": df['regret'].mean() > 0.3,
    "No constant features": (df[CONTEXT_FEATURES].std() > 0).all(),
}

for check, passed in checks.items():
    status = "✅" if passed else "❌"
    print(f"  {status} {check}")

print(f"\nDataset is {'READY' if all(checks.values()) else 'NEEDS FIXES'} for bandit training.")
print("=" * 60)